In [ ]:
%pip install xgboost

In [ ]:
import xgboost

print("Version XGBoost :", xgboost.__version__)

In [ ]:
from xgboost import XGBClassifier

In [ ]:
# ============================================================
# 01 — CHARGEMENT MÉMOIRE-EFFICACE
# ============================================================

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from xgboost import XGBClassifier

PROCESSED_DIR = Path("../data/processed")

print("Répertoire :", PROCESSED_DIR.resolve())

# ------------------------------------------------------------
# Charger uniquement le train pour commencer
# ------------------------------------------------------------

X_train = sparse.load_npz(
    PROCESSED_DIR / "X_train_final.npz"
)

y_train = joblib.load(
    PROCESSED_DIR / "y_train.joblib"
)

scale_pos_weight = joblib.load(
    PROCESSED_DIR / "scale_pos_weight.joblib"
)

print("\n=== TRAIN CHARGÉ ===")

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print(
    "scale_pos_weight :",
    round(scale_pos_weight, 2)
)

In [ ]:
# ============================================================
# 02 — MODÈLE XGBOOST BASELINE
# ============================================================

model_baseline = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",

    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=1,
    reg_alpha=0,
    reg_lambda=1,

    scale_pos_weight=scale_pos_weight,

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)

print("=== MODÈLE XGBOOST ===\n")
print(model_baseline)

In [ ]:
# ============================================
# CELLULE 3 — ENTRAÎNEMENT DU MODÈLE XGBOOST
# ============================================

print("Début de l'entraînement...")

model_baseline.fit(
    X_train,
    y_train,
    verbose=True
)

print("\n=== ENTRAÎNEMENT TERMINÉ ===")

In [ ]:
# ============================================
# CELLULE 4 — ÉVALUATION SUR LA VALIDATION
# ============================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Charger la validation
X_validation = sparse.load_npz(
    PROCESSED_DIR / "X_validation_final.npz"
)

y_validation = joblib.load(
    PROCESSED_DIR / "y_validation.joblib"
)

print("=== VALIDATION CHARGÉE ===")
print("X_validation :", X_validation.shape)
print("y_validation :", y_validation.shape)

# Probabilités de fraude
y_validation_proba = model_baseline.predict_proba(
    X_validation
)[:, 1]

# Métriques principales
auc_pr = average_precision_score(
    y_validation,
    y_validation_proba
)

auc_roc = roc_auc_score(
    y_validation,
    y_validation_proba
)

print("\n=== PERFORMANCES ===")
print(f"PR-AUC  : {auc_pr:.6f}")
print(f"ROC-AUC : {auc_roc:.6f}")

In [ ]:
# ============================================
# CELLULE 5 — ANALYSE DES SEUILS DE DÉCISION
# ============================================

from sklearn.metrics import precision_recall_fscore_support

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

results = []

for threshold in thresholds:
    y_pred = (y_validation_proba >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_validation,
        y_pred,
        average="binary",
        zero_division=0
    )

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "fraud_detected": y_pred.sum()
    })

threshold_results = pd.DataFrame(results)

print("=== PERFORMANCES SELON LE SEUIL ===")
display(threshold_results)

In [ ]:
# ============================================
# CELLULE 6 — MATRICE DE CONFUSION
# ============================================

from sklearn.metrics import confusion_matrix

threshold = 0.30

y_pred_030 = (y_validation_proba >= threshold).astype(int)

cm = confusion_matrix(
    y_validation,
    y_pred_030
)

tn, fp, fn, tp = cm.ravel()

print("=== MATRICE DE CONFUSION — SEUIL 0.30 ===\n")

print(f"Vrais négatifs (TN) : {tn:,}")
print(f"Faux positifs (FP) : {fp:,}")
print(f"Faux négatifs (FN) : {fn:,}")
print(f"Vrais positifs (TP) : {tp:,}")

print("\n=== TAUX ===")

false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)

print(f"Taux de faux positifs : {false_positive_rate:.6%}")
print(f"Taux de faux négatifs : {false_negative_rate:.6%}")

In [ ]:
# ============================================
# CELLULE 7 — DISTRIBUTION DES SCORES DE FRAUDE
# ============================================

# Séparer les scores selon la vraie classe
scores_normal = y_validation_proba[y_validation == 0]
scores_fraud = y_validation_proba[y_validation == 1]

print("=== SCORES DES TRANSACTIONS NORMALES ===")
print(pd.Series(scores_normal).describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
))

print("\n=== SCORES DES TRANSACTIONS FRAUDULEUSES ===")
print(pd.Series(scores_fraud).describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
))

In [ ]:
# ============================================
# CELLULE 8 — ANALYSE APPROVED / OTP / BLOCKED
# ============================================

thresholds_otp = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
thresholds_block = [0.70, 0.80, 0.90, 0.95, 0.97]

results_3_levels = []

for otp_threshold in thresholds_otp:
    for block_threshold in thresholds_block:

        if otp_threshold >= block_threshold:
            continue

        # Zones de décision
        approved = y_validation_proba < otp_threshold
        otp_required = (
            (y_validation_proba >= otp_threshold) &
            (y_validation_proba < block_threshold)
        )
        blocked = y_validation_proba >= block_threshold

        # Vraies classes
        normal = y_validation == 0
        fraud = y_validation == 1

        # Fraudes détectées par OTP ou blocage
        fraud_detected = np.sum(
            fraud & (y_validation_proba >= otp_threshold)
        )

        total_fraud = np.sum(fraud)

        # Fraudes manquées
        fraud_missed = np.sum(
            fraud & approved
        )

        # Transactions normales envoyées vers OTP
        normal_otp = np.sum(
            normal & otp_required
        )

        # Transactions normales bloquées
        normal_blocked = np.sum(
            normal & blocked
        )

        results_3_levels.append({
            "otp_threshold": otp_threshold,
            "block_threshold": block_threshold,
            "fraud_detection_rate": fraud_detected / total_fraud,
            "fraud_missed_rate": fraud_missed / total_fraud,
            "normal_otp_count": normal_otp,
            "normal_blocked_count": normal_blocked,
            "total_otp": np.sum(otp_required),
            "total_blocked": np.sum(blocked)
        })

results_3_levels = pd.DataFrame(results_3_levels)

print("=== SCÉNARIOS À TROIS NIVEAUX ===")

display(
    results_3_levels.sort_values(
        ["fraud_detection_rate", "normal_blocked_count"],
        ascending=[False, True]
    ).head(20)
)

In [ ]:
# ============================================
# CELLULE 9 — LOGIQUE MÉTIER SANS BLOCAGE DIRECT
# ============================================

otp_threshold = 0.30

approved = y_validation_proba < otp_threshold
otp_required = y_validation_proba >= otp_threshold

normal = y_validation == 0
fraud = y_validation == 1

# Transactions normales qui déclenchent un OTP
normal_otp = np.sum(normal & otp_required)

# Fraudes qui déclenchent un OTP
fraud_otp = np.sum(fraud & otp_required)

# Fraudes qui passent directement
fraud_approved = np.sum(fraud & approved)

# Transactions normales approuvées directement
normal_approved = np.sum(normal & approved)

print("=== SIMULATION DE LA POLITIQUE MÉTIER ===")
print(f"Seuil OTP : {otp_threshold}")

print("\n--- TRANSACTIONS NORMALES ---")
print(f"Approuvées directement : {normal_approved:,}")
print(f"OTP requis             : {normal_otp:,}")

print("\n--- TRANSACTIONS FRAUDULEUSES ---")
print(f"OTP requis             : {fraud_otp:,}")
print(f"Passées directement    : {fraud_approved:,}")

print("\n=== TAUX ===")
print(
    f"Transactions normales envoyées vers OTP : "
    f"{normal_otp / np.sum(normal):.4%}"
)

print(
    f"Fraudes interceptées par OTP : "
    f"{fraud_otp / np.sum(fraud):.4%}"
)

In [ ]:
# ============================================
# CELLULE 10 — CHOIX DU SEUIL OTP
# ============================================

otp_thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

threshold_analysis = []

normal = y_validation == 0
fraud = y_validation == 1

total_normal = np.sum(normal)
total_fraud = np.sum(fraud)

for threshold in otp_thresholds:

    otp = y_validation_proba >= threshold

    normal_otp = np.sum(normal & otp)
    fraud_otp = np.sum(fraud & otp)

    threshold_analysis.append({
        "otp_threshold": threshold,
        "normal_otp_count": normal_otp,
        "normal_otp_rate": normal_otp / total_normal,
        "fraud_intercepted": fraud_otp,
        "fraud_detection_rate": fraud_otp / total_fraud,
        "fraud_missed": total_fraud - fraud_otp,
        "fraud_missed_rate": (total_fraud - fraud_otp) / total_fraud
    })

threshold_analysis = pd.DataFrame(threshold_analysis)

print("=== COMPARAISON DES SEUILS OTP ===")

display(
    threshold_analysis.style.format({
        "otp_threshold": "{:.2f}",
        "normal_otp_rate": "{:.4%}",
        "fraud_detection_rate": "{:.4%}",
        "fraud_missed_rate": "{:.4%}"
    })
)


In [ ]:
# ============================================
# CELLULE 11 — SIMULATION DU PARCOURS OTP
# ============================================

otp_threshold = 0.30

# Transaction nécessitant un OTP
otp_required = y_validation_proba >= otp_threshold

# Hypothèse de simulation :
# - transaction normale + OTP => succès
# - transaction frauduleuse + OTP => échec

normal = y_validation == 0
fraud = y_validation == 1

# Transactions normales envoyées vers OTP
normal_otp = normal & otp_required

# Fraudes envoyées vers OTP
fraud_otp = fraud & otp_required

# Après authentification
normal_otp_success = np.sum(normal_otp)
fraud_otp_failure = np.sum(fraud_otp)

# Fraudes qui passent directement sans OTP
fraud_direct_approved = np.sum(
    fraud & ~otp_required
)

print("=== SIMULATION DU PARCOURS OTP ===")

print("\n--- TRANSACTIONS NORMALES ---")
print(f"OTP requis       : {np.sum(normal_otp):,}")
print(f"OTP réussi       : {normal_otp_success:,}")

print("\n--- TRANSACTIONS FRAUDULEUSES ---")
print(f"OTP requis       : {np.sum(fraud_otp):,}")
print(f"OTP échoué       : {fraud_otp_failure:,}")
print(f"Passées sans OTP : {fraud_direct_approved:,}")

print("\n=== RÉSULTAT FINAL ===")

print(
    f"Fraudes empêchées par l'OTP : "
    f"{fraud_otp_failure / np.sum(fraud):.4%}"
)

print(
    f"Fraudes passant directement : "
    f"{fraud_direct_approved / np.sum(fraud):.4%}"
)

In [ ]:
# ============================================
# CELLULE 12 — ÉVALUATION FINALE SUR LE TEST 2019
# ============================================

# Charger le jeu de test
X_test = sparse.load_npz(
    PROCESSED_DIR / "X_test_final.npz"
)

y_test = joblib.load(
    PROCESSED_DIR / "y_test.joblib"
)

print("=== TEST CHARGÉ ===")
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

# Prédictions probabilistes
y_test_proba = model_baseline.predict_proba(
    X_test
)[:, 1]

# Métriques globales
test_pr_auc = average_precision_score(
    y_test,
    y_test_proba
)

test_roc_auc = roc_auc_score(
    y_test,
    y_test_proba
)

# Seuil métier choisi sur la validation
otp_threshold = 0.30

y_test_pred = (
    y_test_proba >= otp_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_test_pred
).ravel()

print("\n=== PERFORMANCES FINALES — TEST 2019 ===")
print(f"PR-AUC  : {test_pr_auc:.6f}")
print(f"ROC-AUC : {test_roc_auc:.6f}")

print("\n=== SEUIL OTP = 0.30 ===")
print(f"TN : {tn:,}")
print(f"FP : {fp:,}")
print(f"FN : {fn:,}")
print(f"TP : {tp:,}")

print("\n=== TAUX ===")

print(
    f"Précision : "
    f"{tp / (tp + fp):.4%}"
)

print(
    f"Rappel : "
    f"{tp / (tp + fn):.4%}"
)

print(
    f"Taux de faux positifs : "
    f"{fp / (fp + tn):.6%}"
)

In [ ]:
# ============================================
# TEST — FRAUDES RÉELLES ET SCORES XGBOOST
# ============================================

# Indices des vraies fraudes
fraud_indices = np.where(y_test == 1)[0]

# Scores correspondants
fraud_scores = y_test_proba[fraud_indices]

# Construire un tableau
fraud_analysis = pd.DataFrame({
    "index": fraud_indices,
    "fraud_probability": fraud_scores
})

# Trier par score croissant
fraud_analysis = fraud_analysis.sort_values(
    "fraud_probability"
).reset_index(drop=True)

print("=== 20 FRAUDES AVEC LES PLUS FAIBLES SCORES ===")
display(
    fraud_analysis.head(20)
)

print("\n=== 20 FRAUDES AVEC LES PLUS FORTS SCORES ===")
display(
    fraud_analysis.tail(20).sort_values(
        "fraud_probability",
        ascending=False
    )
)

In [ ]:
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("y_test_proba :", y_test_proba.shape)

In [ ]:
# ============================================
# ANALYSE DES FRAUDES À TRÈS FAIBLE SCORE
# ============================================

fraud_scores = y_test_proba[y_test == 1]

print("Nombre total de fraudes :", len(fraud_scores))

print("\n=== FRAUDES AVEC SCORE < 0.05 ===")
print(
    np.sum(fraud_scores < 0.05),
    "sur",
    len(fraud_scores)
)

print("\n=== FRAUDES AVEC SCORE < 0.10 ===")
print(
    np.sum(fraud_scores < 0.10),
    "sur",
    len(fraud_scores)
)

print("\n=== FRAUDES AVEC SCORE < 0.20 ===")
print(
    np.sum(fraud_scores < 0.20),
    "sur",
    len(fraud_scores)
)

print("\n=== FRAUDES AVEC SCORE < 0.30 ===")
print(
    np.sum(fraud_scores < 0.30),
    "sur",
    len(fraud_scores)
)

print("\n=== POURCENTAGES ===")

for threshold in [0.05, 0.10, 0.20, 0.30]:
    count = np.sum(fraud_scores < threshold)
    percentage = count / len(fraud_scores)

    print(
        f"Score < {threshold:.2f} : "
        f"{count:,} fraudes "
        f"({percentage:.2%})"
    )

In [ ]:
# ============================================
# LES 22 FRAUDES AVEC SCORE < 0.05
# ============================================

low_score_mask = (
    (y_test == 1)
    & (y_test_proba < 0.05)
)

low_score_indices = np.where(low_score_mask)[0]

print(
    "Nombre de fraudes avec score < 0.05 :",
    len(low_score_indices)
)

print("\nIndices :")
print(low_score_indices)

In [ ]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

MODEL_DATA_FILE = PROCESSED_DIR / "model_data.parquet"

model_data = pd.read_parquet(MODEL_DATA_FILE)

model_data["date"] = pd.to_datetime(model_data["date"])

test_data = model_data[
    model_data["date"].dt.year == 2019
].copy()

print("model_data :", model_data.shape)
print("test_data  :", test_data.shape)
print("y_test     :", y_test.shape)

In [ ]:
import numpy as np

print("Taille test_data :", len(test_data))
print("Taille y_test    :", len(y_test))

print("Fraudes test_data :", test_data["is_fraud"].sum())
print("Fraudes y_test    :", y_test.sum())

print("Même ordre ?      :", np.array_equal(
    test_data["is_fraud"].to_numpy(),
    y_test
))

print("Même distribution :", test_data["is_fraud"].sum() == y_test.sum())

In [ ]:
# Vérifier les premiers indices des fraudes
fraud_positions_test_data = np.where(test_data["is_fraud"].to_numpy() == 1)[0][:20]
fraud_positions_y_test = np.where(y_test == 1)[0][:20]

print("Premières positions des fraudes dans test_data :")
print(fraud_positions_test_data)

print("\nPremières positions des fraudes dans y_test :")
print(fraud_positions_y_test)


In [ ]:
print("Index de test_data :")
print(test_data.index[:20].to_list())

print("\nIndex de y_test :")
print(type(y_test))

print("\nPremières valeurs y_test :")
print(y_test[:20])

In [ ]:
test_indices = test_data.index
y_indices = y_test.index

print("Nombre d'indices test_data :", len(test_indices))
print("Nombre d'indices y_test     :", len(y_indices))

print("\nPremiers indices test_data :")
print(test_indices[:20].to_list())

print("\nPremiers indices y_test :")
print(y_indices[:20].to_list())

print("\nIndices communs :", len(test_indices.intersection(y_indices)))

print("Indices identiques et même ordre :",
      test_indices.equals(y_indices))

In [ ]:
# Vérifier si les index de y_test correspondent bien à des lignes de model_data

print("Nombre d'index y_test présents dans model_data :")
print(y_test.index.isin(model_data.index).sum())

print("\nNombre total d'index y_test :")
print(len(y_test))

# Reconstruire le vrai test_data à partir des index de y_test
test_data_correct = model_data.loc[y_test.index].copy()

print("\nTaille test_data_correct :")
print(test_data_correct.shape)

print("\nFraudes test_data_correct :")
print(test_data_correct["is_fraud"].sum())

print("\nMême cible que y_test ?")
print(
    np.array_equal(
        test_data_correct["is_fraud"].to_numpy(),
        y_test.to_numpy()
    )
)

In [ ]:
print("Index model_data uniques :", model_data.index.is_unique)
print("Index y_test uniques     :", y_test.index.is_unique)

print("\nAnnées des lignes correspondant aux index de y_test :")
print(
    model_data.loc[y_test.index, "date"]
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nRépartition des labels dans model_data sur ces index :")
print(
    model_data.loc[y_test.index, "is_fraud"]
    .value_counts()
    .sort_index()
)

print("\nRépartition des labels dans y_test :")
print(
    y_test.value_counts()
    .sort_index()
)

In [ ]:
# Reproduire exactement l'ordre du notebook 2

model_data_rebuilt = (
    model_data
    .sort_values(["date", "id"])
    .reset_index(drop=True)
)

# Recréer exactement le jeu de test
test_data_correct = model_data_rebuilt[
    model_data_rebuilt["year"] == 2019
].copy()

print("Taille test_data_correct :", test_data_correct.shape)
print("Fraudes test_data_correct :", test_data_correct["is_fraud"].sum())

print("\nDates :")
print("Min :", test_data_correct["date"].min())
print("Max :", test_data_correct["date"].max())

print("\nComparaison avec y_test :")
print(
    "Même cible ?",
    np.array_equal(
        test_data_correct["is_fraud"].to_numpy(),
        y_test.to_numpy()
    )
)

In [ ]:
# ============================================================
# FRAUDES MANQUÉES — ANALYSE CORRECTE
# ============================================================

missed_mask = (
    (y_test.to_numpy() == 1)
    & (y_test_proba < 0.30)
)

missed_frauds = test_data_correct.loc[missed_mask].copy()

missed_frauds["fraud_probability"] = y_test_proba[missed_mask]

print("Nombre de fraudes manquées :", len(missed_frauds))

print("\n=== PREMIÈRES FRAUDES MANQUÉES ===")

cols = [
    "id",
    "date",
    "amount_numeric",
    "use_chip",
    "merchant_state",
    "card_brand",
    "card_type",
    "has_chip",
    "mcc",
    "credit_score",
    "client_transaction_count_1h",
    "client_transaction_count_24h",
    "amount_to_income",
    "amount_to_credit_limit",
    "error_count",
    "fraud_probability",
    "is_fraud"
]

display(
    missed_frauds[
        cols
    ]
    .sort_values("fraud_probability")
    .head(20)
)

In [ ]:
# ============================================================
# PROFIL DES FRAUDES MANQUÉES
# ============================================================

print("=== PROFIL DES 433 FRAUDES MANQUÉES ===")

print("\n--- Score de fraude ---")
print(
    missed_frauds["fraud_probability"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\n--- Use chip ---")
print(
    missed_frauds["use_chip"]
    .value_counts()
)

print("\n--- Merchant state ---")
print(
    missed_frauds["merchant_state"]
    .value_counts()
    .head(15)
)

print("\n--- MCC ---")
print(
    missed_frauds["mcc"]
    .value_counts()
    .head(15)
)

print("\n--- Card brand ---")
print(
    missed_frauds["card_brand"]
    .value_counts()
)

print("\n--- Card type ---")
print(
    missed_frauds["card_type"]
    .value_counts()
)

print("\n--- Erreurs ---")
print(
    missed_frauds["error_count"]
    .value_counts()
)

print("\n--- Statistiques montant ---")
print(
    missed_frauds["amount_numeric"]
    .describe()
)

In [ ]:
# ============================================================
# COMPARAISON : FRAUDES DÉTECTÉES VS FRAUDES MANQUÉES
# ============================================================

detected_mask = (
    (y_test.to_numpy() == 1)
    & (y_test_proba >= 0.30)
)

detected_frauds = test_data_correct.loc[detected_mask].copy()
detected_frauds["fraud_probability"] = y_test_proba[detected_mask]

print("Fraudes détectées :", len(detected_frauds))
print("Fraudes manquées  :", len(missed_frauds))


# ------------------------------------------------------------
# Comparaison des variables numériques
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Fraudes détectées": detected_frauds[
        [
            "amount_numeric",
            "credit_score",
            "client_transaction_count_1h",
            "client_transaction_count_24h",
            "amount_to_income",
            "amount_to_credit_limit",
            "error_count"
        ]
    ].median(),

    "Fraudes manquées": missed_frauds[
        [
            "amount_numeric",
            "credit_score",
            "client_transaction_count_1h",
            "client_transaction_count_24h",
            "amount_to_income",
            "amount_to_credit_limit",
            "error_count"
        ]
    ].median()
})

print("\n=== MÉDIANES ===")
display(comparison)


# ------------------------------------------------------------
# Comparaison des variables catégorielles
# ------------------------------------------------------------

for col in [
    "use_chip",
    "merchant_state",
    "card_brand",
    "card_type",
    "mcc"
]:
    print(f"\n=== {col.upper()} ===")

    detected_pct = (
        detected_frauds[col]
        .value_counts(normalize=True)
        .mul(100)
        .head(10)
        .rename("Détectées (%)")
    )

    missed_pct = (
        missed_frauds[col]
        .value_counts(normalize=True)
        .mul(100)
        .head(10)
        .rename("Manquées (%)")
    )

    comp = pd.concat(
        [detected_pct, missed_pct],
        axis=1
    ).fillna(0)

    display(comp)

In [ ]:
# ============================================================
# TAUX DE DÉTECTION DES FRAUDES PAR MCC
# ============================================================

analysis_data = test_data_correct.copy()

analysis_data["fraud_probability"] = y_test_proba
analysis_data["predicted_fraud"] = (
    y_test_proba >= 0.30
).astype(int)

# On garde uniquement les vraies fraudes
fraud_by_mcc = analysis_data[
    analysis_data["is_fraud"] == 1
].groupby("mcc").agg(
    fraud_count=("is_fraud", "size"),
    fraud_detected=("predicted_fraud", "sum"),
    mean_score=("fraud_probability", "mean")
)

fraud_by_mcc["detection_rate"] = (
    fraud_by_mcc["fraud_detected"]
    / fraud_by_mcc["fraud_count"]
)

fraud_by_mcc["missed_count"] = (
    fraud_by_mcc["fraud_count"]
    - fraud_by_mcc["fraud_detected"]
)

fraud_by_mcc = fraud_by_mcc.sort_values(
    ["fraud_count", "detection_rate"],
    ascending=[False, True]
)

print("=== DÉTECTION DES FRAUDES PAR MCC ===")

display(
    fraud_by_mcc[
        [
            "fraud_count",
            "fraud_detected",
            "missed_count",
            "detection_rate",
            "mean_score"
        ]
    ].head(25)
)

In [ ]:
# ============================================================
# ANALYSE DE PLUSIEURS SEUILS OTP
# ============================================================

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

threshold_results = []

for threshold in thresholds:
    y_pred = (y_test_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    otp_rate = y_pred.mean()

    threshold_results.append({
        "threshold": threshold,
        "OTP_required": int(y_pred.sum()),
        "OTP_rate": otp_rate,
        "frauds_detected": tp,
        "frauds_missed": fn,
        "fraud_recall": recall,
        "false_positives": fp,
        "false_positive_rate": fpr,
        "precision": precision
    })

threshold_results = pd.DataFrame(threshold_results)

print("=== COMPARAISON DES SEUILS ===")

display(
    threshold_results.style.format({
        "threshold": "{:.2f}",
        "OTP_rate": "{:.2%}",
        "fraud_recall": "{:.2%}",
        "false_positive_rate": "{:.4%}",
        "precision": "{:.2%}"
    })
)

In [ ]:
# ============================================================
# COMPARAISON DÉTAILLÉE DES FRAUDES DÉTECTÉES / MANQUÉES
# ============================================================

numeric_compare = [
    "amount_numeric",
    "credit_score",
    "client_transaction_count_1h",
    "client_transaction_count_24h",
    "client_amount_24h",
    "amount_deviation_from_client",
    "amount_change_ratio",
    "time_since_previous_transaction",
    "amount_to_income",
    "amount_to_credit_limit",
    "client_absolute_amount_24h",
    "card_transaction_count_24h",
    "card_amount_24h",
    "merchant_transaction_count_24h",
    "merchant_amount_24h",
    "error_count"
]

comparison_stats = pd.DataFrame({
    "Detectees_median": detected_frauds[numeric_compare].median(),
    "Manquees_median": missed_frauds[numeric_compare].median(),
    "Detectees_mean": detected_frauds[numeric_compare].mean(),
    "Manquees_mean": missed_frauds[numeric_compare].mean()
})

comparison_stats["ecart_median_%"] = (
    (
        comparison_stats["Manquees_median"]
        - comparison_stats["Detectees_median"]
    )
    / comparison_stats["Detectees_median"].replace(0, np.nan)
    * 100
)

display(
    comparison_stats.sort_values(
        "ecart_median_%",
        key=lambda x: x.abs(),
        ascending=False
    )
)

In [ ]:
from pathlib import Path

print("Dossier de travail :", Path.cwd())
print("Dossier des données :", PROCESSED_DIR)
print("Le dossier existe :", PROCESSED_DIR.exists())
print("Fichiers présents :")

if PROCESSED_DIR.exists():
    for f in PROCESSED_DIR.iterdir():
        print(f.name)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent.parent
print("Dossier du projet :", PROJECT_DIR)

for fichier in PROJECT_DIR.rglob("feature_names.joblib"):
    print("Fichier trouvé :", fichier)

In [ ]:
# ============================================================
# IMPORTANCE DES VARIABLES XGBOOST
# ============================================================

import joblib

FEATURE_NAMES_PATH = PROJECT_DIR / "ml" / "models" / "feature_names.joblib"

feature_names = joblib.load(FEATURE_NAMES_PATH)

print("Nombre de features :", len(feature_names))
print("Exemples :", feature_names[:10])


feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model_baseline.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

print("\n=== TOP 30 FEATURES XGBOOST ===")

display(
    feature_importance.head(30)
)

In [ ]:
# ============================================
# CELLULE 13 — ANALYSE DES FRAUDES MANQUÉES
# ============================================

# Fraudes correctement détectées
fraud_detected_scores = y_test_proba[
    (y_test == 1) & (y_test_proba >= 0.30)
]

# Fraudes manquées
fraud_missed_scores = y_test_proba[
    (y_test == 1) & (y_test_proba < 0.30)
]

print("=== ANALYSE DES FRAUDES MANQUÉES ===")

print(f"\nFraudes détectées : {len(fraud_detected_scores):,}")
print(f"Fraudes manquées  : {len(fraud_missed_scores):,}")

print("\n=== SCORE DES FRAUDES DÉTECTÉES ===")
print(
    pd.Series(fraud_detected_scores).describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

print("\n=== SCORE DES FRAUDES MANQUÉES ===")
print(
    pd.Series(fraud_missed_scores).describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

In [ ]:
# ============================================
# CELLULE 14 — IMPORTANCE DES VARIABLES
# ============================================

# Importance selon le gain
importance_gain = model_baseline.get_booster().get_score(
    importance_type="gain"
)

# Transformer en DataFrame
feature_importance = pd.DataFrame(
    list(importance_gain.items()),
    columns=["feature", "gain"]
)

# Trier
feature_importance = feature_importance.sort_values(
    "gain",
    ascending=False
).reset_index(drop=True)

print("=== TOP 30 DES VARIABLES LES PLUS IMPORTANTES ===")

display(
    feature_importance.head(30)
)

In [ ]:
# ============================================
# CELLULE 15 — RECONSTRUCTION DES NOMS FEATURES
# ============================================

from sklearn.preprocessing import OneHotEncoder

# ------------------------------------------------------------
# 1. Charger les données nécessaires
# ------------------------------------------------------------

MODEL_DATA_FILE = PROCESSED_DIR / "model_data.parquet"

model_data = pd.read_parquet(MODEL_DATA_FILE)

# ------------------------------------------------------------
# 2. Reconstituer exactement le découpage temporel
# ------------------------------------------------------------

model_data["date"] = pd.to_datetime(model_data["date"])

train_data = model_data[
    model_data["date"].dt.year <= 2017
].copy()

# ------------------------------------------------------------
# 3. Variables catégorielles utilisées lors de la préparation
# ------------------------------------------------------------

CATEGORICAL_FEATURES = [
    "use_chip",
    "merchant_state",
    "card_brand",
    "card_type",
    "has_chip",
    "mcc"
]

# ------------------------------------------------------------
# 4. Recréer l'encodeur EXACTEMENT comme dans 02
# ------------------------------------------------------------

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

categorical_encoder.fit(
    train_data[CATEGORICAL_FEATURES]
)

# ------------------------------------------------------------
# 5. Noms des variables numériques
# ------------------------------------------------------------

NUMERIC_FEATURES = [
    "amount_numeric",
    "has_error",
    "error_count",
    "current_age",
    "yearly_income_numeric",
    "total_debt_numeric",
    "credit_score",
    "num_credit_cards",
    "credit_limit_numeric",
    "previous_amount",
    "previous_amount_missing",
    "time_since_previous_transaction",
    "amount_difference",
    "amount_change_ratio",
    "client_previous_avg_amount",
    "amount_deviation_from_client",
    "client_transaction_count_1h",
    "client_transaction_count_24h",
    "client_amount_24h",
    "amount_abs",
    "client_absolute_amount_24h",
    "card_transaction_count_24h",
    "card_amount_24h",
    "merchant_transaction_count_24h",
    "merchant_amount_24h",
    "year",
    "month",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "is_night",
    "debt_to_income",
    "amount_to_income",
    "amount_to_credit_limit",
    "amount_to_credit_limit_missing",
    "bad_pin",
    "bad_cvv",
    "bad_card_number",
    "bad_expiration",
    "bad_zipcode",
    "insufficient_balance",
    "technical_glitch",
    "amount_log",
    "amount_difference_log",
    "amount_change_ratio_log",
    "amount_deviation_from_client_log",
    "amount_to_income_log",
    "amount_to_credit_limit_log",
    "client_amount_24h_log",
    "client_absolute_amount_24h_log",
    "card_amount_24h_log",
    "merchant_amount_24h_log",
    "zip_missing"
]

# ------------------------------------------------------------
# 6. Noms des variables One-Hot
# ------------------------------------------------------------

categorical_feature_names = list(
    categorical_encoder.get_feature_names_out(
        CATEGORICAL_FEATURES
    )
)

# ------------------------------------------------------------
# 7. Liste finale dans le même ordre que X_train_final
# ------------------------------------------------------------

feature_names = (
    NUMERIC_FEATURES
    + categorical_feature_names
)

# ------------------------------------------------------------
# 8. Contrôles
# ------------------------------------------------------------

print("=== CONTRÔLE DES FEATURES ===")

print("Features numériques       :", len(NUMERIC_FEATURES))
print("Features catégorielles    :", len(categorical_feature_names))
print("Features totales          :", len(feature_names))
print("Colonnes X_train          :", X_train.shape[1])

assert len(NUMERIC_FEATURES) == 53
assert len(categorical_feature_names) == 310
assert len(feature_names) == X_train.shape[1] == 363

print("\n=== PREMIÈRES FEATURES ===")

for i, name in enumerate(feature_names[:20]):
    print(f"f{i} -> {name}")

print("\n=== DERNIÈRES FEATURES ===")

for i, name in enumerate(
    feature_names[-10:],
    start=len(feature_names) - 10
):
    print(f"f{i} -> {name}")

In [ ]:
# ============================================
# CELLULE 16 — IMPORTANCE AVEC NOMS RÉELS
# ============================================

# Reprendre les importances calculées à la cellule précédente
feature_importance_named = feature_importance.copy()

# Remplacer f0, f1, ..., f362 par les vrais noms
feature_importance_named["feature_name"] = (
    feature_importance_named["feature"]
    .str.replace("f", "", regex=False)
    .astype(int)
    .map(lambda i: feature_names[i])
)

# Réorganiser
feature_importance_named = feature_importance_named[
    ["feature", "feature_name", "gain"]
].sort_values(
    "gain",
    ascending=False
).reset_index(drop=True)

print("=== TOP 30 DES VARIABLES LES PLUS IMPORTANTES ===")

display(
    feature_importance_named.head(30)
)

In [ ]:
# ============================================
# CELLULE 17 — ANALYSE DES VARIABLES IMPORTANTES
# ============================================

# Recréer zip_missing exactement comme lors de la préparation
model_data["zip_missing"] = model_data["zip"].isna().astype("int8")

# ------------------------------------------------------------
# 1. zip_missing
# ------------------------------------------------------------

print("=== FRAUDE SELON zip_missing ===")

zip_analysis = (
    model_data.groupby("zip_missing")["is_fraud"]
    .agg(
        transactions="count",
        frauds="sum",
        fraud_rate="mean"
    )
    .reset_index()
)

display(
    zip_analysis.style.format({
        "fraud_rate": "{:.6%}"
    })
)

# ------------------------------------------------------------
# 2. use_chip
# ------------------------------------------------------------

print("\n=== FRAUDE SELON use_chip ===")

chip_analysis = (
    model_data.groupby("use_chip")["is_fraud"]
    .agg(
        transactions="count",
        frauds="sum",
        fraud_rate="mean"
    )
    .reset_index()
)

display(
    chip_analysis.style.format({
        "fraud_rate": "{:.6%}"
    })
)

# ------------------------------------------------------------
# 3. MCC
# ------------------------------------------------------------

print("\n=== TOP MCC SELON LE TAUX DE FRAUDE ===")

mcc_analysis = (
    model_data.groupby("mcc")["is_fraud"]
    .agg(
        transactions="count",
        frauds="sum",
        fraud_rate="mean"
    )
    .reset_index()
)

# Garder uniquement les MCC suffisamment représentés
mcc_analysis_filtered = mcc_analysis[
    mcc_analysis["transactions"] >= 100
].sort_values(
    "fraud_rate",
    ascending=False
)

display(
    mcc_analysis_filtered.head(20).style.format({
        "fraud_rate": "{:.6%}"
    })
)

In [ ]:
# ============================================
# CELLULE 18 — STABILITÉ DES SIGNAUX FRAUDE
# ============================================

# Ajouter l'année à partir de la date
model_data["analysis_year"] = model_data["date"].dt.year

# ------------------------------------------------------------
# 1. zip_missing par année
# ------------------------------------------------------------

print("=== ZIP MISSING PAR ANNÉE ===")

zip_year = (
    model_data.groupby(
        ["analysis_year", "zip_missing"]
    )["is_fraud"]
    .agg(
        transactions="count",
        frauds="sum",
        fraud_rate="mean"
    )
    .reset_index()
)

display(
    zip_year.style.format({
        "fraud_rate": "{:.6%}"
    })
)

# ------------------------------------------------------------
# 2. use_chip par année
# ------------------------------------------------------------

print("\n=== USE_CHIP PAR ANNÉE ===")

chip_year = (
    model_data.groupby(
        ["analysis_year", "use_chip"]
    )["is_fraud"]
    .agg(
        transactions="count",
        frauds="sum",
        fraud_rate="mean"
    )
    .reset_index()
)

display(
    chip_year.style.format({
        "fraud_rate": "{:.6%}"
    })
)

# ------------------------------------------------------------
# 3. MCC — évolution des principaux MCC
# ------------------------------------------------------------

top_mcc = (
    mcc_analysis_filtered
    .head(10)["mcc"]
    .tolist()
)

print("\n=== ÉVOLUTION DES 10 MCC LES PLUS RISQUÉS ===")

mcc_year = (
    model_data[
        model_data["mcc"].isin(top_mcc)
    ]
    .groupby(
        ["analysis_year", "mcc"]
    )["is_fraud"]
    .agg(
        transactions="count",
        frauds="sum",
        fraud_rate="mean"
    )
    .reset_index()
)

display(
    mcc_year.style.format({
        "fraud_rate": "{:.6%}"
    })
)

In [ ]:
# ============================================
# CELLULE 19 — SAUVEGARDE DU MODÈLE
# ============================================

from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "xgboost_fraud_model_v1.json"

# Sauvegarde native XGBoost
model_baseline.save_model(MODEL_PATH)

print("=== MODÈLE SAUVEGARDÉ ===")
print("Chemin :", MODEL_PATH)
print("Taille :", MODEL_PATH.stat().st_size / (1024 * 1024), "MB")

In [ ]:
# ============================================
# CELLULE 20 — SAUVEGARDE DES MÉTADONNÉES
# ============================================

import joblib

# ------------------------------------------------------------
# 1. Noms des 363 features
# ------------------------------------------------------------

FEATURE_NAMES_PATH = MODEL_DIR / "feature_names.joblib"

joblib.dump(
    feature_names,
    FEATURE_NAMES_PATH
)

# ------------------------------------------------------------
# 2. Métadonnées métier du modèle
# ------------------------------------------------------------

MODEL_METADATA = {
    "model_name": "xgboost_fraud_model",
    "model_version": "v1.0.0",
    "model_type": "XGBClassifier",
    "threshold_otp": 0.30,
    "feature_count": len(feature_names),
    "numeric_feature_count": len(NUMERIC_FEATURES),
    "categorical_feature_count": len(categorical_feature_names),
    "training_period": "2010-2017",
    "validation_period": "2018",
    "test_period": "2019",
    "test_pr_auc": float(test_pr_auc),
    "test_roc_auc": float(test_roc_auc)
}

METADATA_PATH = MODEL_DIR / "model_metadata.joblib"

joblib.dump(
    MODEL_METADATA,
    METADATA_PATH
)

print("=== MÉTADONNÉES SAUVEGARDÉES ===")
print("Feature names :", FEATURE_NAMES_PATH)
print("Model metadata:", METADATA_PATH)

print("\n=== CONFIGURATION ===")
for key, value in MODEL_METADATA.items():
    print(f"{key}: {value}")

In [ ]:
# ============================================
# CELLULE 21 — VÉRIFICATION DES ARTEFACTS
# ============================================

from pathlib import Path
import joblib

print("=== VÉRIFICATION DES FICHIERS DU MODÈLE ===\n")

files_to_check = {
    "Modèle XGBoost": MODEL_PATH,
    "Noms des features": FEATURE_NAMES_PATH,
    "Métadonnées": METADATA_PATH
}

for name, path in files_to_check.items():

    exists = path.exists()

    print(f"{name}")
    print(f"  Chemin  : {path}")
    print(f"  Existe  : {exists}")

    if exists:
        print(
            f"  Taille  : "
            f"{path.stat().st_size / (1024 * 1024):.3f} MB"
        )

    print()

# Vérification du chargement
print("=== TEST DE CHARGEMENT ===")

test_features = joblib.load(FEATURE_NAMES_PATH)
test_metadata = joblib.load(METADATA_PATH)

print(
    f"Nombre de features chargées : {len(test_features)}"
)

print(
    f"Version du modèle : "
    f"{test_metadata['model_version']}"
)

print(
    f"Seuil OTP : "
    f"{test_metadata['threshold_otp']}"
)

assert len(test_features) == 363
assert test_metadata["threshold_otp"] == 0.30

print("\n✅ Tous les artefacts sont valides.")

In [ ]:
# ============================================
# CELLULE 22 — TEST DU MODÈLE RECHARGÉ
# ============================================

from xgboost import XGBClassifier

# Créer une nouvelle instance
loaded_model = XGBClassifier()

# Charger le modèle sauvegardé
loaded_model.load_model(MODEL_PATH)

print("=== MODÈLE RECHARGÉ ===")
print(loaded_model)

# Prendre 5 transactions du test
X_sample = X_test[:5]

# Calcul des scores
sample_scores = loaded_model.predict_proba(X_sample)[:, 1]

print("\n=== TEST DE SCORING ===")

for i, score in enumerate(sample_scores, start=1):
    decision = (
        "OTP_REQUIRED"
        if score >= 0.30
        else "APPROVED"
    )

    print(
        f"Transaction {i} | "
        f"Score = {score:.6f} | "
        f"Décision = {decision}"
    )

# Vérifier que le modèle rechargé donne les mêmes résultats
original_scores = model_baseline.predict_proba(X_sample)[:, 1]

difference = np.max(
    np.abs(original_scores - sample_scores)
)

print("\n=== VÉRIFICATION ===")
print(f"Différence maximale : {difference:.12f}")

assert difference < 1e-10

print("✅ Le modèle sauvegardé produit les mêmes scores.")

In [ ]:
# ============================================
# CELLULE 23 — ARTEFACTS DE PRÉPROCESSING
# ============================================

print("=== VÉRIFICATION DES ARTEFACTS ===\n")

# Liste des fichiers présents dans ../models
for path in sorted(MODEL_DIR.iterdir()):

    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)

        print(
            f"{path.name:<40} "
            f"{size_mb:.3f} MB"
        )

print("\n=== OBJETS ACTUELLEMENT DISPONIBLES ===")

print(
    "categorical_encoder :",
    "OK" if "categorical_encoder" in globals()
    else "ABSENT"
)

print(
    "NUMERIC_FEATURES :",
    "OK" if "NUMERIC_FEATURES" in globals()
    else "ABSENT"
)

print(
    "feature_names :",
    "OK" if "feature_names" in globals()
    else "ABSENT"
)

In [ ]:
print([v for v in globals() if "metadata" in v.lower()])

In [ ]:
print("MODEL_METADATA :", type(MODEL_METADATA))
print("METADATA_PATH :", METADATA_PATH)
print("test_metadata :", type(test_metadata))

print("\nContenu de MODEL_METADATA :")
print(MODEL_METADATA)

print("\nContenu de test_metadata :")
print(test_metadata)

In [ ]:
# ============================================
# CELLULE 24 — SAUVEGARDE DE L'ENCODEUR
# ============================================

ENCODER_PATH = MODEL_DIR / "categorical_encoder.joblib"

joblib.dump(
    categorical_encoder,
    ENCODER_PATH
)

print("=== ENCODEUR SAUVEGARDÉ ===")
print("Chemin :", ENCODER_PATH)
print(
    "Taille :",
    f"{ENCODER_PATH.stat().st_size / (1024 * 1024):.3f} MB"
)

# Vérification immédiate
loaded_encoder = joblib.load(ENCODER_PATH)

print("\n=== VÉRIFICATION ===")
print(
    "Nombre de features encodées :",
    len(loaded_encoder.get_feature_names_out())
)

assert len(
    loaded_encoder.get_feature_names_out()
) == 310

print("✅ Encodeur sauvegardé et vérifié.")

In [ ]:
# ============================================
# CELLULE 25 — TEST COMPLET DU PIPELINE
# ============================================

# ------------------------------------------------------------
# 1. Prendre une transaction réelle du jeu de test
# ------------------------------------------------------------

test_data = model_data[
    model_data["date"].dt.year == 2019
].copy()

sample = test_data.iloc[[0]].copy()

print("=== TRANSACTION TEST ===")
print(sample[[
    "id",
    "date",
    "use_chip",
    "merchant_state",
    "card_brand",
    "card_type",
    "has_chip",
    "mcc"
]])

# ------------------------------------------------------------
# 2. Reproduire les traitements de 02_data_preparation
# ------------------------------------------------------------

# --- Historique client ---
sample["previous_amount_missing"] = (
    sample["previous_amount"]
    .isna()
    .astype("int8")
)

sample["previous_amount"] = (
    sample["previous_amount"].fillna(0)
)

sample["amount_difference"] = (
    sample["amount_difference"].fillna(0)
)

sample["client_previous_avg_amount"] = (
    sample["client_previous_avg_amount"].fillna(0)
)

sample["amount_deviation_from_client"] = (
    sample["amount_deviation_from_client"].fillna(0)
)

sample["amount_change_ratio"] = (
    sample["amount_change_ratio"].fillna(0)
)

sample["time_since_previous_transaction"] = (
    sample["time_since_previous_transaction"].fillna(-1)
)

# --- Code postal ---
sample["zip_missing"] = (
    sample["zip"]
    .isna()
    .astype("int8")
)

sample["zip"] = (
    sample["zip"].fillna(-1)
)

# --- Ratio montant / limite ---
sample["amount_to_credit_limit_missing"] = (
    sample["amount_to_credit_limit"]
    .isna()
    .astype("int8")
)

sample["amount_to_credit_limit"] = (
    sample["amount_to_credit_limit"].fillna(0)
)

# --- Variables catégorielles ---
sample["errors"] = (
    sample["errors"].fillna("No Error")
)

sample["merchant_state"] = (
    sample["merchant_state"].fillna("Unknown")
)

# ------------------------------------------------------------
# 3. Recréer les transformations logarithmiques
# ------------------------------------------------------------

def safe_log1p(series):
    return np.log1p(
        series.where(series >= 0)
    )

def signed_log1p(series):
    return (
        np.sign(series)
        * np.log1p(series.abs())
    )

sample["amount_log"] = signed_log1p(
    sample["amount_numeric"]
)

sample["amount_difference_log"] = safe_log1p(
    sample["amount_difference"]
)

sample["amount_change_ratio_log"] = safe_log1p(
    sample["amount_change_ratio"]
)

sample["amount_deviation_from_client_log"] = safe_log1p(
    sample["amount_deviation_from_client"]
)

sample["amount_to_income_log"] = signed_log1p(
    sample["amount_to_income"]
)

sample["amount_to_credit_limit_log"] = signed_log1p(
    sample["amount_to_credit_limit"]
)

sample["client_amount_24h_log"] = signed_log1p(
    sample["client_amount_24h"]
)

sample["client_absolute_amount_24h_log"] = safe_log1p(
    sample["client_absolute_amount_24h"]
)

sample["card_amount_24h_log"] = signed_log1p(
    sample["card_amount_24h"]
)

sample["merchant_amount_24h_log"] = signed_log1p(
    sample["merchant_amount_24h"]
)

# ------------------------------------------------------------
# 4. Variables numériques
# ------------------------------------------------------------

sample_num = sample[NUMERIC_FEATURES].copy()

sample_num = sample_num.replace(
    [np.inf, -np.inf],
    np.nan
)

print("\n=== CONTRÔLE NUMÉRIQUE ===")
print("Features numériques :", sample_num.shape[1])
print("Valeurs NaN          :", sample_num.isna().sum().sum())

# ------------------------------------------------------------
# 5. Variables catégorielles
# ------------------------------------------------------------

sample_cat = sample[CATEGORICAL_FEATURES].copy()

# ------------------------------------------------------------
# 6. One-Hot Encoding
# ------------------------------------------------------------

sample_cat_encoded = loaded_encoder.transform(
    sample_cat
)

print(
    "Features encodées    :",
    sample_cat_encoded.shape[1]
)

# ------------------------------------------------------------
# 7. Assemblage final
# ------------------------------------------------------------

sample_num_sparse = sparse.csr_matrix(
    sample_num.to_numpy(dtype=np.float32)
)

sample_final = sparse.hstack(
    [
        sample_num_sparse,
        sample_cat_encoded
    ],
    format="csr"
)

print("\n=== MATRICE FINALE ===")
print("Dimensions :", sample_final.shape)

assert sample_final.shape == (1, 363)

# ------------------------------------------------------------
# 8. Scoring
# ------------------------------------------------------------

sample_score = loaded_model.predict_proba(
    sample_final
)[:, 1][0]

threshold_otp = MODEL_METADATA["threshold_otp"]
sample_decision = (
    "APPROVED"
    if sample_score < threshold_otp
    else "OTP_REQUIRED"
)

print("\n=== SCORING ===")
print(f"Score de fraude : {sample_score:.6f}")
print(f"Seuil OTP       : {threshold_otp:.2f}")
print(f"Décision        : {sample_decision}")

print("\n Pipeline complet validé.")

In [ ]:
# ============================================
# CELLULE 26 — FONCTION DE SCORING
# ============================================

def score_transaction(
    transaction_num,
    transaction_cat,
    model,
    encoder,
    threshold=0.30
):
    """
    Transforme une transaction et retourne
    son score de fraude et sa décision.
    """

    # -----------------------------
    # Copie pour éviter les effets
    # de bord
    # -----------------------------
    transaction_num = transaction_num.copy()
    transaction_cat = transaction_cat.copy()

    # -----------------------------
    # Valeurs manquantes
    # -----------------------------
    transaction_num["previous_amount_missing"] = (
        transaction_num["previous_amount"]
        .isna()
        .astype("int8")
    )

    transaction_num["previous_amount"] = (
        transaction_num["previous_amount"]
        .fillna(0)
    )

    transaction_num["amount_difference"] = (
        transaction_num["amount_difference"]
        .fillna(0)
    )

    transaction_num["amount_change_ratio"] = (
        transaction_num["amount_change_ratio"]
        .fillna(0)
    )

    transaction_num["client_previous_avg_amount"] = (
        transaction_num["client_previous_avg_amount"]
        .fillna(0)
    )

    transaction_num["amount_deviation_from_client"] = (
        transaction_num["amount_deviation_from_client"]
        .fillna(0)
    )

    transaction_num["time_since_previous_transaction"] = (
        transaction_num["time_since_previous_transaction"]
        .fillna(-1)
    )

    transaction_num["zip_missing"] = (
        transaction_num["zip"]
        .isna()
        .astype("int8")
    )

    transaction_num["zip"] = (
        transaction_num["zip"]
        .fillna(-1)
    )

    transaction_num["amount_to_credit_limit_missing"] = (
        transaction_num["amount_to_credit_limit"]
        .isna()
        .astype("int8")
    )

    transaction_num["amount_to_credit_limit"] = (
        transaction_num["amount_to_credit_limit"]
        .fillna(0)
    )

    # -----------------------------
    # Transformations logarithmiques
    # -----------------------------
    transaction_num["amount_log"] = signed_log1p(
        transaction_num["amount_numeric"]
    )

    transaction_num["amount_difference_log"] = safe_log1p(
        transaction_num["amount_difference"]
    )

    transaction_num["amount_change_ratio_log"] = safe_log1p(
        transaction_num["amount_change_ratio"]
    )

    transaction_num["amount_deviation_from_client_log"] = safe_log1p(
        transaction_num["amount_deviation_from_client"]
    )

    transaction_num["amount_to_income_log"] = signed_log1p(
        transaction_num["amount_to_income"]
    )

    transaction_num["amount_to_credit_limit_log"] = signed_log1p(
        transaction_num["amount_to_credit_limit"]
    )

    transaction_num["client_amount_24h_log"] = signed_log1p(
        transaction_num["client_amount_24h"]
    )

    transaction_num["client_absolute_amount_24h_log"] = safe_log1p(
        transaction_num["client_absolute_amount_24h"]
    )

    transaction_num["card_amount_24h_log"] = signed_log1p(
        transaction_num["card_amount_24h"]
    )

    transaction_num["merchant_amount_24h_log"] = signed_log1p(
        transaction_num["merchant_amount_24h"]
    )

    # -----------------------------
    # Catégorielles
    # -----------------------------
    transaction_cat["errors"] = (
        transaction_cat["errors"]
        .fillna("No Error")
    )

    transaction_cat["merchant_state"] = (
        transaction_cat["merchant_state"]
        .fillna("Unknown")
    )

    # -----------------------------
    # Sélection des 53 numériques
    # -----------------------------
    transaction_num = transaction_num[
        NUMERIC_FEATURES
    ]

    # -----------------------------
    # Encodage des 6 catégorielles
    # -----------------------------
    encoded_cat = encoder.transform(
        transaction_cat[CATEGORICAL_FEATURES]
    )

    # -----------------------------
    # Assemblage des 363 features
    # -----------------------------
    numeric_sparse = sparse.csr_matrix(
        transaction_num.to_numpy(dtype=np.float32)
    )

    final_features = sparse.hstack(
        [
            numeric_sparse,
            encoded_cat
        ],
        format="csr"
    )

    # -----------------------------
    # Vérification
    # -----------------------------
    if final_features.shape[1] != 363:
        raise ValueError(
            f"Nombre de features incorrect : "
            f"{final_features.shape[1]} au lieu de 363"
        )

    # -----------------------------
    # Score
    # -----------------------------
    score = float(
        model.predict_proba(final_features)[0, 1]
    )

    # -----------------------------
    # Décision
    # -----------------------------
    decision = (
        "APPROVED"
        if score < threshold
        else "OTP_REQUIRED"
    )

    return {
        "fraud_score": score,
        "decision": decision
    }


print("✅ Fonction score_transaction créée.")

In [ ]:
# ============================================
# CELLULE 27 — TEST DE LA FONCTION DE SCORING
# ============================================

# Prendre la même transaction de test
sample = test_data.iloc[[0]].copy()

# Séparer les variables numériques et catégorielles
sample_num = sample.copy()
sample_cat = sample.copy()

# Scoring avec notre fonction
result = score_transaction(
    transaction_num=sample_num,
    transaction_cat=sample_cat,
    model=loaded_model,
    encoder=loaded_encoder,
    threshold_otp = MODEL_METADATA["threshold_otp"]
)

print("=== RÉSULTAT DU SCORING ===")

print(f"Score de fraude : {result['fraud_score']:.6f}")
print(f"Décision        : {result['decision']}")

print("\n=== INFORMATIONS ===")
print(f"Transaction ID  : {sample['id'].iloc[0]}")
print(f"Date            : {sample['date'].iloc[0]}")

assert result["decision"] in [
    "APPROVED",
    "OTP_REQUIRED"
]

assert 0 <= result["fraud_score"] <= 1

print("\n✅ Fonction de scoring validée.")

In [ ]:
# ============================================
# CELLULE 28 — PRÉPARATION DU PACKAGE MODÈLE
# ============================================

import shutil

# Dossier de destination pour le service de scoring
SERVICE_MODEL_DIR = Path(
    "../../services/fraud-scoring-service/models"
)

SERVICE_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Fichiers nécessaires au service
MODEL_FILES = [
    MODEL_PATH,
    ENCODER_PATH,
    FEATURE_NAMES_PATH,
    METADATA_PATH
]

print("=== COPIE DES ARTEFACTS ===\n")

for source in MODEL_FILES:

    destination = SERVICE_MODEL_DIR / source.name

    shutil.copy2(
        source,
        destination
    )

    print(
        f"✓ {source.name} → {destination}"
    )

print("\n=== CONTENU DU DOSSIER ===")

for path in sorted(SERVICE_MODEL_DIR.iterdir()):
    if path.is_file():
        print(
            f"{path.name:<40} "
            f"{path.stat().st_size / (1024 * 1024):.3f} MB"
        )

print("\n✅ Package modèle prêt pour le fraud-scoring-service.")

In [ ]:
import xgboost

print("Version XGBoost :", xgboost.__version__)

In [ ]:
import sys
import xgboost

print("=== NOTEBOOK ===")
print("Python :", sys.executable)
print("XGBoost :", xgboost.__version__)
print("XGBoost path :", xgboost.__file__)

# Visualisations de l'entraînement et de l'évaluation XGBoost

Cette section rassemble les graphiques de performance du modèle principal. Les courbes ROC/Precision-Recall et les résultats finaux utilisent le test 2019, tandis que l'analyse des seuils d'origine reste fondée sur la validation 2018. Le seuil opérationnel 0,30 est celui choisi sur la validation.


In [ ]:
# 1. Courbe précision-rappel — test 2019
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, average_precision_score

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_proba)
test_ap_plot = average_precision_score(y_test, y_test_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall_curve, precision_curve, label=f"XGBoost — PR-AUC = {test_ap_plot:.4f}")
ax.axhline(y_test.mean(), linestyle="--", label=f"Prévalence = {y_test.mean():.4%}")
ax.set_title("Courbe précision-rappel — test 2019")
ax.set_xlabel("Rappel")
ax.set_ylabel("Précision")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 2. Courbe ROC — test 2019
from sklearn.metrics import roc_curve, roc_auc_score

fpr_curve, tpr_curve, _ = roc_curve(y_test, y_test_proba)
roc_plot = roc_auc_score(y_test, y_test_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_curve, tpr_curve, label=f"XGBoost — ROC-AUC = {roc_plot:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", label="Classifieur aléatoire")
ax.set_title("Courbe ROC — test 2019")
ax.set_xlabel("Taux de faux positifs")
ax.set_ylabel("Taux de vrais positifs")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Matrice de confusion au seuil opérationnel 0,30 — test 2019
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_test_pred_plot = (y_test_proba >= 0.30).astype(int)
cm_plot = confusion_matrix(y_test, y_test_pred_plot, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm_plot, display_labels=["Légitime", "Fraude"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, values_format=",", colorbar=False)
ax.set_title("Matrice de confusion — test 2019, seuil 0,30")
plt.tight_layout()
plt.show()


In [ ]:
# 5. Compromis précision / rappel / F1 selon le seuil (table des seuils test)
if "threshold_results" in globals() and isinstance(threshold_results, pd.DataFrame) and "threshold" in threshold_results.columns:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for col, label in [("precision", "Précision"), ("fraud_recall", "Rappel"), ("recall", "Rappel"), ("f1_score", "F1")]:
        if col in threshold_results.columns:
            ax.plot(threshold_results["threshold"], threshold_results[col], marker="o", label=label)
    ax.axvline(0.30, color="black", linestyle="--", label="Seuil retenu 0,30")
    ax.set_title("Métriques selon le seuil — analyse des seuils")
    ax.set_xlabel("Seuil")
    ax.set_ylabel("Score")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
# 6. Top 20 des variables les plus importantes selon le gain XGBoost
if "importance_gain" in globals() and len(importance_gain):
    imp_plot = pd.DataFrame(list(importance_gain.items()), columns=["feature", "gain"]).sort_values("gain").tail(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(imp_plot["feature"].astype(str), imp_plot["gain"])
    ax.set_title("Top 20 des variables les plus importantes — gain XGBoost")
    ax.set_xlabel("Gain")
    plt.tight_layout()
    plt.show()
elif "feature_importance" in globals() and isinstance(feature_importance, pd.DataFrame) and len(feature_importance):
    value_col = "gain" if "gain" in feature_importance.columns else "importance"
    name_col = "feature" if "feature" in feature_importance.columns else feature_importance.columns[0]
    imp_plot = feature_importance.sort_values(value_col).tail(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(imp_plot[name_col].astype(str), imp_plot[value_col])
    ax.set_title("Top 20 des variables importantes — XGBoost")
    ax.set_xlabel(value_col)
    plt.tight_layout()
    plt.show()


In [ ]:
# 7. Fraudes détectées et manquées au seuil 0,30
fraud_counts = pd.Series({
    "Fraudes détectées": int(((np.asarray(y_test) == 1) & (y_test_proba >= 0.30)).sum()),
    "Fraudes manquées": int(((np.asarray(y_test) == 1) & (y_test_proba < 0.30)).sum()),
})
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(fraud_counts.index, fraud_counts.values)
ax.set_title("Fraudes détectées vs manquées — test 2019")
ax.set_ylabel("Nombre de fraudes")
for b, v in zip(bars, fraud_counts.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
# 8. Taux de détection par MCC — les catégories avec le plus de fraudes
if "fraud_by_mcc" in globals() and isinstance(fraud_by_mcc, pd.DataFrame) and "detection_rate" in fraud_by_mcc.columns:
    mcc_plot = fraud_by_mcc.sort_values("fraud_count", ascending=False).head(15).sort_values("detection_rate")
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(mcc_plot.index.astype(str), mcc_plot["detection_rate"]*100)
    ax.set_title("Taux de détection des fraudes par MCC — principaux volumes")
    ax.set_xlabel("Fraudes détectées (%)")
    ax.set_ylabel("MCC")
    plt.tight_layout()
    plt.show()
